## Experiment

In [ ]:
import os
for k,v in os.environ.items():
    if k.startswith("QUANTDATA_"):
        print(f"{k}={v}")
        

In [ ]:

%load_ext autoreload
%autoreload 1

#%aimport quantdata
#%aimport quantdata.biz_calendar
#%aimport quantdata.widb
#%aimport quantdata.widb.client
#%aimport quantdata.widb.day
#%aimport quantdata.widb.arbfree
#%aimport quantdata.widb.options_filter
#%aimport quantdata.utils.backbone
#%aimport quantdata.sanos.sanos_tools
#%aimport quantdata.sanos.arbfree
#%aimport quantdata.sanos.sanos
#%aimport quantdata.dysanos
%aimport quantdata.dysanos.grid.torch_market
%aimport quantdata.dysanos.grid.torch_pricer
%aimport quantdata.dysanos.grid.torch_model
%aimport quantdata.dysanos.grid.torch_gfx 
%aimport quantdata.dysanos.grid.torch_train 
%aimport quantdata.dysanos.grid.torch_simulator
%aimport quantdata.dysanos.grid.torch_universe
%aimport quantdata.dysanos.grid.torch_drift_change
%aimport quantdata.dysanos.grid.torch_det_measure_change
%aimport quantdata.dysanos.grid.torch_all

from quantdata.dysanos.grid.torch_all import run_experiment, DysanosExperiment
import gc
gc.collect()

experiment_config = DysanosExperiment(num_samples=250_000, num_steps=20)

ex = run_experiment(
    experiment_config,
    plot_sampler=True,
    plot_det_measure_change=True)

print("*** READY ***")

In [ ]:
from quantdata.dysanos.grid.torch_drift_change import DualRiskNeutralDeepHedgingConfig, FeedForwardConfig, _notebook_name, train, TrainConfig, VizConfig, BaseEnvironment, Context
import torch as torch
import numpy as np

mdl_config = DualRiskNeutralDeepHedgingConfig( 
    label=_notebook_name(),
    group="dysanos", 
    user_version="0.1", 
    utility="powexp", 
    lmbda=1.0, 
    delta_hedged=False, 
    dual_weight=0.1,
    tert_weight=0.1,
    cost = 0.00, # 1 bps
    cost_std = 1./np.sqrt( experiment_config.num_samples ),
    boot_strap=1,
    boot_strap_dual=1,
    boot_strap_tert=1,
    use_fixed=True,
    in_config=FeedForwardConfig(
        num_depth = 2,
        num_width = 40,
        activation = "LeakyReLU",
        use_bias = True
    ),
    pf_config=None,
    tau_block_size=2,
)

import gc
model_RN_output = 0
gc.collect()
print(torch.cuda.memory_allocated() / 1024**3, "GiB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GiB reserved")

def run():
    trn_config = TrainConfig( num_epochs=200_000, 
                              pcnt_validation=0.,
                              lr=1E-2, final_lr=1E-6, weight_decay=1e-4, batch_size=None, compile=True, epoch_interval=100 )
    viz_config = VizConfig()
    model_RN   = train(
                    ex.universe,
                    ex.sampler, 
                    ref_dQdP=ex.dQdP_dS,
                    mdl_config=mdl_config, 
                    trn_config=trn_config, 
                    viz_config=viz_config, restart=False, just_load="no",
                   env=BaseEnvironment(device="cpu",debug=False), verbose=Context.all )
    gc.collect()
    return model_RN

model_RN_output = run()
